# سیستم چندعاملی تحلیل و اتوماسیون فرآیندهای سازمانی
# Multi-Agent System for the Analysis & Automation of Organisational Processes

**Generative AI — Final Project**

---

## ۱. مرور کلی و هدف پروژه — Overview and objective

In a *Systems & Methods* (سیستم‌ها و روش‌ها) unit the hardest question is not
"what does the procedure say?" and not "what do the numbers say?" — it is the
**gap between the two**: between the approved procedure written in the company's
documentation and the way the work is really executed, day after day, in the
ticketing system.

This project builds an intelligent multi-agent system, based on **Agentic RAG**
and **Tool Calling**, that

1. **understands** the organisation's documentation,
2. **reads** the real data out of the operational systems,
3. **discovers** the bottlenecks, and
4. **proposes** concrete automation measures — under the supervision of a human
   systems manager who can veto and redirect any proposal.

---

## ۲. معماری سیستم و جریان کاری — Architecture and workflow

The system is a **LangGraph decision graph** with four nodes and one control
mechanism:

| # | Node | Role | Mechanism | Output |
|---|------|------|-----------|--------|
| ۱ | `documentation_agent` | تحلیل‌گر مستندات — QA & process expert | **Agentic RAG** over a vector store of SOPs and flowcharts | the approved SLA and steps (*what should be*) |
| ۲ | `diagnostic_agent` | بازرس عملیات — systems data analyst | **Tool Calling** against a mock Jira/ServiceDesk API | real metrics (*what is*) |
| ۳ | `process_engineer` | مهندس فرآیند — S&M architect | gap analysis + remediation catalogue | an **Action Plan** with automation scripts |
| ۴ | `human_review` | تاییدیه مدیر سیستم‌ها — you | **Checkpointer + interrupt** (Human-in-the-Loop) | approval, or feedback that triggers a new revision |

```mermaid
graph TD
    START([START]) --> A[Node 1 · documentation_agent<br/>Agentic RAG over the SOP corpus]
    A -->|standard_sop : what SHOULD be| B[Node 2 · diagnostic_agent<br/>Tool Calling on ServiceDesk/Jira]
    B -->|actual_metrics : what IS| C[Node 3 · process_engineer<br/>gap analysis + Action Plan]
    C --> P{{"⏸ CHECKPOINTER<br/>interrupt_before"}}
    P --> D[Node 4 · human_review<br/>Human-in-the-Loop]
    D -->|approved| E([END])
    D -->|human_feedback → re-plan| C
```

---

## How this notebook is organised

The complete system is implemented as a small, properly layered Python package
under `src/process_automation/`, written against the **SOLID** principles. This
notebook is the *executable report*: every class the brief asks for is printed
here, in full, straight from the source file — so the code you read below is
literally the code that runs, with no copy-paste drift between the two.

| Section | Brief requirement |
|---------|-------------------|
| 3 | مرحله ۱ — the `WorkflowState` TypedDict |
| 4 | مرحله ۲ — the `@tool` monitoring functions |
| 5–8 | the four agent nodes |
| 9 | مرحله ۳ — conditional edges and the checkpointer |
| 10 | `.get_graph().draw_mermaid_png()` |
| 11 | سناریوی اول — bottleneck found, automation accepted |
| 12 | سناریوی دوم — human intervention and course correction |
| 13 | test suite and the SOLID design review |

---
## 0. Setup

The system runs **fully offline**: the retrieval, the gap analysis and the
remediation catalogue are deterministic, so this notebook produces the same
output on any machine, with no API key and no network access.

A hosted chat model is optional — see section 13.5 — and is used only to
rewrite the management summary in nicer prose, never to compute a number.

In [1]:
# Only two runtime dependencies are required.
# %pip install -q langgraph langchain-core

import logging
import sys
from pathlib import Path

# Make the project package importable from the notebook.
PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Route the agents' logs to the notebook output: the brief asks to *see* the
# RAG extraction, the tool calls and the moment the graph stops for approval.
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    stream=sys.stdout,
    force=True,
)

import langgraph
import process_automation

print("python           :", sys.version.split()[0])
print("process_automation:", process_automation.__version__)
print("source tree      :", SRC)

python           : 3.11.15
process_automation: 1.0.0
source tree      : /home/user/Multi_Agent_Automation/src


In [2]:
import inspect
from IPython.display import Markdown, display


def show_source(obj, note: str = "") -> None:
    """Print the real source of a class/function, straight from the package.

    Used throughout this notebook so that every piece of code the brief asks
    for is visible *and* is guaranteed to be the code that actually executes.
    """
    source = inspect.getsource(obj)
    path = Path(inspect.getsourcefile(obj)).relative_to(PROJECT_ROOT)
    header = f"##### `{obj.__name__}` — `{path}`"
    if note:
        header += f"\n\n{note}"
    display(Markdown(f"{header}\n\n```python\n{source}\n```"))


def rule(title: str) -> None:
    """Visual separator for the scenario runs."""
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

---
## ۳. مرحله ۱ — تعریف وضعیت (State)

The brief specifies the exact `TypedDict` that carries information between the
nodes. It is reproduced **verbatim** in `process_automation/state.py`.

Everything the runtime additionally needs (revision counter, constraints,
audit trail…) lives in a **subclass** with `total=False`, so the required
contract is extended, never modified — the Open/Closed Principle applied to a
data structure.

Two of the extra keys use `Annotated[list[str], operator.add]`. That installs a
**reducer**: when a node returns `{"trace": [...]}` LangGraph *appends* instead
of overwriting, which turns those fields into an append-only audit log.

In [3]:
from process_automation.state import ProcessAnalysisState, WorkflowState, initial_state

show_source(WorkflowState, "The state required by the brief, unchanged.")
show_source(ProcessAnalysisState, "The extension actually compiled into the graph.")

##### `WorkflowState` — `src/process_automation/state.py`

The state required by the brief, unchanged.

```python
class WorkflowState(TypedDict):
    """Shared memory of the multi-agent graph."""

    process_name: str       # نام فرآیند درخواستی - the requested process
    standard_sop: str       # استاندارد استخراج شده (خروجی RAG) - Node 1 output
    actual_metrics: dict    # دیتای واقعی استخراج شده (خروجی Tools) - Node 2 output
    identified_gaps: list   # شکاف‌ها و گلوگاه‌های شناسایی شده - Node 3 finding
    proposed_solution: str  # راهکار نهایی اتوماسیون - Node 3 output
    human_feedback: str     # بازخورد کاربر انسانی برای اصلاح - Node 4 input

```

##### `ProcessAnalysisState` — `src/process_automation/state.py`

The extension actually compiled into the graph.

```python
class ProcessAnalysisState(WorkflowState, total=False):
    """``WorkflowState`` + the bookkeeping the runtime needs.

    ``total=False`` marks every key below as optional, so a run can still be
    started with nothing but ``process_name``.

    ``Annotated[list, operator.add]`` installs a *reducer*: when a node
    returns ``{"trace": ["..."]}`` LangGraph appends instead of overwriting,
    which is what turns these two fields into an append-only audit log.
    """

    #: True once the systems manager has approved the current plan.
    approved: bool
    #: 1 for the first plan, 2 after the first round of human feedback, ...
    revision: int
    #: Machine-readable constraints distilled from the human feedback.
    constraints: list[dict[str, Any]]
    #: Structured mirror of ``proposed_solution`` (gaps + proposals + code).
    action_plan: dict[str, Any]
    #: Chunks the RAG node actually used, for traceability of the extraction.
    retrieved_documents: list[dict[str, Any]]
    #: Canonical process name resolved by the RAG node from the user's wording.
    resolved_process: str
    #: Identifier of the SOP document the standard was extracted from.
    sop_document_id: str
    #: Structured mirror of ``standard_sop`` (SLA + typed steps), so Node 3 can
    #: compare numbers instead of re-parsing prose.
    standard_profile: dict[str, Any]
    #: Append-only execution log (RAG lookups, tool calls, routing decisions).
    trace: Annotated[list[str], operator.add]
    #: Append-only history of everything the human reviewer said.
    feedback_log: Annotated[list[str], operator.add]

```

In [4]:
# A fresh run starts with every key initialised - no defensive `.get()` noise
# inside the nodes, and a perfectly reproducible starting point.
show_source(initial_state)
initial_state("VPN access provisioning for new personnel")

##### `initial_state` — `src/process_automation/state.py`

```python
def initial_state(process_name: str) -> ProcessAnalysisState:
    """Build a fully-populated starting state for one analysis run.

    Initialising every key up front keeps the nodes free of ``state.get(...)``
    defensive noise and makes a run reproducible.
    """
    return ProcessAnalysisState(
        process_name=process_name,
        standard_sop="",
        actual_metrics={},
        identified_gaps=[],
        proposed_solution="",
        human_feedback="",
        approved=False,
        revision=0,
        constraints=[],
        action_plan={},
        retrieved_documents=[],
        resolved_process="",
        sop_document_id="",
        standard_profile={},
        trace=[],
        feedback_log=[],
    )

```

{'process_name': 'VPN access provisioning for new personnel',
 'standard_sop': '',
 'actual_metrics': {},
 'identified_gaps': [],
 'proposed_solution': '',
 'human_feedback': '',
 'approved': False,
 'revision': 0,
 'constraints': [],
 'action_plan': {},
 'retrieved_documents': [],
 'resolved_process': '',
 'sop_document_id': '',
 'standard_profile': {},
 'trace': [],
 'feedback_log': []}

---
## ۴. مرحله ۲ — ساخت ابزارهای مانیتورینگ (Tool Creation)

The Diagnostic Agent must **call a tool**, exactly as it would call a real
analytics endpoint — it must never read a file itself.

Two tools are exposed, both decorated with `@tool`, both taking a process name
and returning a dictionary:

* `fetch_process_metrics` — the headline KPIs (average resolution time, ticket
  volume, reopened tickets, common errors);
* `fetch_stage_timings` — *where* the time goes, stage by stage.

The tools are produced by a **factory** that closes over an injected dataset
rather than reaching for a module-level global. That is what makes them
unit-testable and lets two graphs run side by side against different backends
(Dependency Inversion).

In [5]:
from process_automation.infrastructure.tools import (
    ServiceDeskDataset,
    ServiceDeskMetricsProvider,
    ServiceDeskToolFactory,
)

show_source(
    ServiceDeskToolFactory,
    "The `@tool`-decorated mock API required by step 2 of the brief.",
)

##### `ServiceDeskToolFactory` — `src/process_automation/infrastructure/tools.py`

The `@tool`-decorated mock API required by step 2 of the brief.

```python
class ServiceDeskToolFactory:
    """Builds the LangChain tools the Diagnostic Agent is allowed to call.

    A factory (rather than module-level tool functions) keeps the dataset
    injected instead of global, which is what makes the tools unit-testable
    and lets several graphs run side by side against different backends.
    """

    def __init__(self, dataset: ServiceDeskDataset) -> None:
        self._dataset = dataset

    def build(self) -> list[BaseTool]:
        """Return the tool objects, each with an LLM-readable docstring."""
        dataset = self._dataset  # captured by the closures below

        @tool
        def fetch_process_metrics(process_name: str) -> dict:
            """Return the headline KPIs measured for an organisational process.

            Use this tool to learn how a process performs *in reality*: the
            average end-to-end resolution time in hours, the ticket volume of
            the sample, how many tickets had to be reopened, and the errors
            operators hit most often.

            Args:
                process_name: Name of the process, e.g. "VPN access
                    provisioning for new personnel", or its SOP identifier.
            """
            record = dataset.payload(process_name)
            return {
                "process_name": record["process_name"],
                "sample_period": record["sample_period"],
                "average_resolution_hours": record["average_resolution_hours"],
                "ticket_volume": record["ticket_volume"],
                "reopened_tickets": record["reopened_tickets"],
                "common_errors": list(record["common_errors"]),
            }

        @tool
        def fetch_stage_timings(process_name: str) -> dict:
            """Return the per-stage duration breakdown of a process.

            Use this tool to find *where* the time goes: it reports, for each
            stage of the real workflow, the average hours spent there and the
            share of that time which is pure waiting (queue time).

            Args:
                process_name: Name of the process, or its SOP identifier.
            """
            record = dataset.payload(process_name)
            return {
                "process_name": record["process_name"],
                "stage_timings": [dict(item) for item in record["stage_timings"]],
            }

        return [fetch_process_metrics, fetch_stage_timings]

```

In [6]:
# The mock backend: a JSON export of the ServiceDesk analytics API.
# In production, `ServiceDeskDataset.payload` would issue an HTTP request and
# absolutely nothing above it would change.
dataset = ServiceDeskDataset.from_json(
    PROJECT_ROOT / "src/process_automation/resources/service_desk/metrics.json"
)
tools = ServiceDeskToolFactory(dataset).build()

for tool_ in tools:
    print(f"tool: {tool_.name}")
    print(f"  args   : {list(tool_.args)}")
    print(f"  purpose: {tool_.description.splitlines()[0]}")

print()
print("monitored processes:", ", ".join(dataset.keys))

tool: fetch_process_metrics
  args   : ['process_name']
  purpose: Return the headline KPIs measured for an organisational process.
tool: fetch_stage_timings
  args   : ['process_name']
  purpose: Return the per-stage duration breakdown of a process.

monitored processes: SOP-IT-014, SOP-SEC-021, SOP-IT-008


In [7]:
# A real tool call, exactly as the agent performs it.
tools[0].invoke({"process_name": "VPN access provisioning for new personnel"})

{'process_name': 'VPN access provisioning for new personnel',
 'sample_period': '2024-Q4 (140 tickets)',
 'average_resolution_hours': 42.0,
 'ticket_volume': 140,
 'reopened_tickets': 18,
 'common_errors': ['Wrong remote-access profile selected at intake',
  'Employee ID missing, ticket bounced back to HR',
  'Firewall rule typed against the wrong user group']}

In [8]:
tools[1].invoke({"process_name": "SOP-IT-014"})

{'process_name': 'VPN access provisioning for new personnel',
 'stage_timings': [{'stage': 'Request registration',
   'average_hours': 0.4,
   'waiting_share': 0.1},
  {'stage': 'Direct manager approval',
   'average_hours': 34.5,
   'waiting_share': 0.95},
  {'stage': 'Security policy check',
   'average_hours': 1.6,
   'waiting_share': 0.3},
  {'stage': 'Firewall rule provisioning',
   'average_hours': 4.8,
   'waiting_share': 0.55},
  {'stage': 'User notification and handover',
   'average_hours': 0.7,
   'waiting_share': 0.2}]}

---
## ۵. گره ۱ — تحلیل‌گر مستندات (Documentation Agent · Agentic RAG)

**نقش:** کارشناس تضمین کیفیت و فرآیندها.
**وظیفه:** گرفتن نام یک فرآیند و جستجو در پایگاه دانش برداری حاوی فلوچارت‌ها و
آیین‌نامه‌های متنی شرکت.
**خروجی:** استخراج دقیق SLA و مراحل مصوب فرآیند.

The knowledge base is a folder of Markdown SOP documents with a front-matter
block and a table of approved steps — so **a new process is on-boarded by
dropping a file in the folder**, with no Python change at all.

The retrieval is genuinely *agentic*: it is a two-hop pipeline. First a cosine
search over chunk embeddings; then a **vote over documents** — every retrieved
chunk contributes its score to its source document — before the winning
document's SLA and step table are parsed into typed objects. Voting on
documents rather than trusting the single best chunk is what keeps the
extraction correct when, say, only the policy-notes section matched the query.

In [9]:
from process_automation.infrastructure.embeddings import HashingTfidfEmbeddings
from process_automation.infrastructure.knowledge_base import MarkdownKnowledgeBaseLoader
from process_automation.infrastructure.retrieval import InMemoryVectorStore, StructuredSopParser

knowledge_base = MarkdownKnowledgeBaseLoader(
    PROJECT_ROOT / "src/process_automation/resources/knowledge_base"
).load()

for document in knowledge_base.documents:
    print(f"{document.document_id:<12} SLA={document.sla_hours:>4.1f}h  "
          f"{len(document.steps)} steps  {document.process}")

SOP-IT-008   SLA= 4.0h  5 steps  Level 2 support ticket handling
SOP-SEC-021  SLA= 8.0h  5 steps  Privileged database access provisioning
SOP-IT-014   SLA= 2.0h  5 steps  VPN access provisioning for new personnel


In [10]:
# Build the vector index. The embedder is a dependency-free, deterministic
# hashing TF-IDF model; swapping in OpenAI/HuggingFace embeddings is a one-line
# change in the composition root because both satisfy the same port.
vector_store = InMemoryVectorStore.from_knowledge_base(
    knowledge_base, HashingTfidfEmbeddings(512), min_score=0.05
)
print(f"{vector_store.size} chunks indexed\n")

for chunk in vector_store.retrieve("VPN access for a new employee", top_k=4):
    print(f"  {chunk.score:.3f}  {chunk.document_id:<12} §{chunk.metadata['section']}")

15 chunks indexed

  0.380  SOP-IT-014   §header
  0.339  SOP-IT-014   §1. Purpose and scope
  0.234  SOP-IT-014   §2. Service level agreement
  0.176  SOP-IT-008   §header


In [11]:
# The same query written in Persian resolves to the same document.
for chunk in vector_store.retrieve("تخصیص دسترسی VPN به پرسنل جدید", top_k=3):
    print(f"  {chunk.score:.3f}  {chunk.document_id:<12} §{chunk.metadata['section']}")

  0.541  SOP-IT-014   §header
  0.201  SOP-SEC-021  §header
  0.095  SOP-IT-014   §2. Service level agreement


In [12]:
from process_automation.agents import DocumentationAgent

show_source(DocumentationAgent)

##### `DocumentationAgent` — `src/process_automation/agents/documentation.py`

```python
class DocumentationAgent(AgentNode):
    """Retrieves and structures the approved standard for a process."""

    name = "documentation_agent"

    def __init__(
        self,
        retriever: DocumentRetriever,
        parser: SopParser,
        *,
        top_k: int = 4,
    ) -> None:
        # Both collaborators are *ports*: this agent works unchanged with an
        # in-memory TF-IDF store or with a hosted FAISS/Chroma index.
        super().__init__()
        self._retriever = retriever
        self._parser = parser
        self._top_k = top_k

    def run(self, state: ProcessAnalysisState) -> Mapping[str, Any]:
        query = state["process_name"]
        trace = [self.log(f"RAG lookup for: '{query}' (top_k={self._top_k})")]

        chunks = self._retriever.retrieve(query, top_k=self._top_k)
        for chunk in chunks:
            trace.append(
                self.log(
                    f"  hit score={chunk.score:.3f} doc={chunk.document_id} "
                    f"section='{chunk.metadata.get('section', '?')}'"
                )
            )
        if not chunks:
            trace.append(self.log("  no chunk passed the relevance threshold"))

        standard = self._parser.parse(query, chunks)
        document_id = chunks[0].document_id if chunks else ""

        if standard.sla_hours > 0:
            trace.append(
                self.log(
                    f"extracted SLA={standard.sla_hours:g}h and "
                    f"{len(standard.steps)} approved step(s) from {document_id}"
                )
            )
            controls = standard.mandatory_controls
            if controls:
                trace.append(
                    self.log(
                        "mandatory control(s) found: "
                        + ", ".join(
                            f"{step.title}"
                            f"{' / ' + step.control_reference if step.control_reference else ''}"
                            for step in controls
                        )
                    )
                )
        else:
            trace.append(
                self.log("no approved standard could be extracted for this process")
            )

        return {
            # The plain-text field required by the project brief...
            "standard_sop": standard.to_text(),
            # ...plus the structured mirror Node 3 needs to compute numbers.
            "standard_profile": standard.to_payload(),
            "resolved_process": standard.process_name or query,
            "sop_document_id": document_id,
            "retrieved_documents": [
                {
                    "document_id": chunk.document_id,
                    "section": chunk.metadata.get("section", ""),
                    "score": chunk.score,
                    "excerpt": _excerpt(chunk.text),
                }
                for chunk in chunks
            ],
            "trace": trace,
        }

```

In [13]:
# Run the node in isolation - no graph needed, because it is a plain object
# whose dependencies were injected.
documentation_agent = DocumentationAgent(
    retriever=vector_store, parser=StructuredSopParser(knowledge_base), top_k=4
)
rag_update = documentation_agent.run(
    initial_state("VPN access provisioning for new personnel")
)

print(rag_update["standard_sop"])

RAG lookup for: 'VPN access provisioning for new personnel' (top_k=4)


  hit score=0.553 doc=SOP-IT-014 section='header'


  hit score=0.347 doc=SOP-IT-014 section='2. Service level agreement'


  hit score=0.318 doc=SOP-IT-014 section='1. Purpose and scope'


  hit score=0.251 doc=SOP-IT-014 section='3. Approved steps'


extracted SLA=2h and 5 approved step(s) from SOP-IT-014


Process: VPN access provisioning for new personnel
Documented SLA: 2 hour(s) end-to-end
Approved steps:
  1. Request registration - owner: ServiceDesk intake bot - target: 0.1h - type: automated
  2. Direct manager approval - owner: Direct Manager - target: 0.5h - type: manual_approval
  3. Security policy check - owner: Security Operations rule engine - target: 0.4h - type: automated
  4. Firewall rule provisioning - owner: Network Operations - target: 0.8h - type: manual_provisioning
  5. User notification and handover - owner: ServiceDesk - target: 0.2h - type: automated
Policy notes:
  - The direct manager's approval records business need; it may be delegated to a named deputy but never skipped.
  - Every granted profile must be time-boxed and re-certified every 180 days.
  - Standard remote-access profiles are pre-approved by Information Security, so no individual security sign-off is required for this process.
  - All access decisions are written to the immutable audit log.
Sourc

---
## ۶. گره ۲ — بازرس عملیات (Diagnostic Agent · Tool Caller)

**نقش:** تحلیل‌گر داده‌های سیستمی.
**وظیفه:** فراخوانی ابزارها برای اتصال به APIهای فرضی (شبیه‌سازی دیتای Jira یا
ServiceDesk) و دریافت معیارهای واقعی فرآیند.
**خروجی:** یک گزارش ساختاریافته از وضعیت واقعی سیستم و متریک‌های عملکردی.

Note the agent's dependency: a `MetricsProvider` **port**, not LangChain. The
adapter that invokes the two tools and merges their payloads lives in the
infrastructure layer, so this agent can be tested with a three-line stub.

In [14]:
from process_automation.agents import DiagnosticAgent

show_source(DiagnosticAgent)

##### `DiagnosticAgent` — `src/process_automation/agents/diagnostic.py`

```python
class DiagnosticAgent(AgentNode):
    """Measures reality by calling tools - it never reads a file itself."""

    name = "diagnostic_agent"

    def __init__(self, metrics_provider: MetricsProvider) -> None:
        super().__init__()
        self._provider = metrics_provider

    def run(self, state: ProcessAnalysisState) -> Mapping[str, Any]:
        # Prefer the canonical name resolved by the RAG node: it matches the
        # monitoring system's vocabulary better than the user's free wording.
        target = state.get("resolved_process") or state["process_name"]
        trace = [
            self.log(
                f"calling tool(s) {', '.join(self._provider.tool_names)} "
                f"for process '{target}'"
            )
        ]

        try:
            metrics = self._provider.fetch(target)
        except LookupError as error:
            # A missing process is a *business* outcome, not a crash: the
            # engineer node will report that reality could not be measured.
            trace.append(self.log(f"tool call failed: {error}"))
            return {"actual_metrics": {}, "trace": trace}

        trace.append(
            self.log(
                f"measured average={metrics.average_resolution_hours:g}h over "
                f"{metrics.ticket_volume} ticket(s) [{metrics.sample_period}], "
                f"reopened={metrics.reopened_tickets} ({metrics.reopen_rate:.0%})"
            )
        )
        for timing in metrics.stage_timings:
            trace.append(
                self.log(
                    f"  stage '{timing.stage}': {timing.average_hours:g}h "
                    f"({metrics.share_of_total(timing):.0%} of cycle, "
                    f"{timing.waiting_share:.0%} waiting)"
                )
            )
        bottleneck = metrics.bottleneck
        if bottleneck is not None:
            trace.append(
                self.log(f"slowest stage is '{bottleneck.stage}'")
            )

        return {"actual_metrics": metrics.to_payload(), "trace": trace}

```

In [15]:
metrics_provider = ServiceDeskMetricsProvider(tools)
diagnostic_agent = DiagnosticAgent(metrics_provider)

# The node receives the state the RAG node produced, so it can use the
# *canonical* process name instead of the user's free wording.
state_after_rag = {**initial_state("VPN access provisioning for new personnel"), **rag_update}
diagnostic_update = diagnostic_agent.run(state_after_rag)

metrics = diagnostic_update["actual_metrics"]
print(f"average resolution : {metrics['average_resolution_hours']} h")
print(f"ticket volume      : {metrics['ticket_volume']} ({metrics['sample_period']})")
print(f"reopened           : {metrics['reopened_tickets']} ({metrics['reopen_rate']:.1%})")
print("\nstage breakdown:")
for stage in metrics["stage_timings"]:
    bar = "█" * max(1, round(stage["average_hours"] / 2))
    print(f"  {stage['stage']:<32} {stage['average_hours']:>5.1f} h  {bar}")

calling tool(s) fetch_process_metrics, fetch_stage_timings for process 'VPN access provisioning for new personnel'


measured average=42h over 140 ticket(s) [2024-Q4 (140 tickets)], reopened=18 (13%)


  stage 'Request registration': 0.4h (1% of cycle, 10% waiting)


  stage 'Direct manager approval': 34.5h (82% of cycle, 95% waiting)


  stage 'Security policy check': 1.6h (4% of cycle, 30% waiting)


  stage 'Firewall rule provisioning': 4.8h (11% of cycle, 55% waiting)


  stage 'User notification and handover': 0.7h (2% of cycle, 20% waiting)


slowest stage is 'Direct manager approval'


average resolution : 42.0 h
ticket volume      : 140 (2024-Q4 (140 tickets))
reopened           : 18 (12.9%)

stage breakdown:
  Request registration               0.4 h  █
  Direct manager approval           34.5 h  █████████████████
  Security policy check              1.6 h  █
  Firewall rule provisioning         4.8 h  ██
  User notification and handover     0.7 h  █


---
## ۷. گره ۳ — مهندس فرآیند (Process Engineer Agent)

**نقش:** معمار سیستم‌ها و روش‌ها.
**وظیفه:** مقایسه خروجی گره ۱ (آنچه باید باشد) و گره ۲ (آنچه هست)، شناسایی
انحرافات و گلوگاه‌ها، و ارائه راهکارهای بهینه‌سازی.
**خروجی:** یک Action Plan جامع شامل لیست مشکلات و کدهای/پیشنهادات اتوماسیون.

The agent itself contains **no business logic at all** — it orchestrates three
injected collaborators:

1. a `GapAnalyzer`, a set of independent **rules** that detect the deviations;
2. a `FeedbackInterpreter`, which turns the reviewer's sentence into
   machine-checkable **constraints**;
3. a `SolutionComposer`, which picks a remedy per gap from a **registry of
   strategies** and renders the plan.

This is why scenario 2 needs no special code path: revision 2 is the *same*
call with a longer constraint list.

### ۷.۱ Gap detection — a rule per kind of deviation

Each rule is a separate class. Adding "the process is executed outside business
hours" means writing a new rule and registering it; `GapAnalyzer` never
changes.

In [16]:
from process_automation.domain.gap_analysis import DEFAULT_RULES, GapAnalyzer, SlaBreachRule

print("Active rules:")
for rule_ in DEFAULT_RULES:
    print(f"  {rule_.code:<20} {type(rule_).__name__}")

show_source(SlaBreachRule, "One rule, in full. The other four follow the same contract.")

Active rules:
  SLA_BREACH           SlaBreachRule
  BOTTLENECK           BottleneckRule
  HIGH_REWORK          ReworkRule
  RECURRING_ERROR      RecurringErrorRule
  UNDOCUMENTED_STAGE   UndocumentedStageRule


##### `SlaBreachRule` — `src/process_automation/domain/gap_analysis.py`

One rule, in full. The other four follow the same contract.

```python
class SlaBreachRule(GapRule):
    """The end-to-end duration exceeds the SLA written in the SOP."""

    code = "SLA_BREACH"

    def __init__(self, tolerance: float = 1.1) -> None:
        # ``tolerance`` absorbs normal statistical noise: only a breach above
        # 110% of the SLA (by default) is reported as a real gap.
        self._tolerance = tolerance

    def evaluate(self, context: ComparisonContext) -> Iterable[Gap]:
        if context.sla_ratio <= self._tolerance:
            return ()
        ratio = context.sla_ratio
        severity = (
            Severity.CRITICAL if ratio >= 5
            else Severity.HIGH if ratio >= 2
            else Severity.MEDIUM
        )
        return (
            Gap(
                code=self.code,
                title="End-to-end duration breaches the documented SLA",
                severity=severity,
                evidence=(
                    f"SOP target is {context.standard.sla_hours:g}h but the measured "
                    f"average is {context.metrics.average_resolution_hours:g}h "
                    f"({ratio:.1f}x the SLA, +{context.sla_overrun_hours:g}h)."
                ),
                lost_hours=context.sla_overrun_hours,
            ),
        )

```

In [17]:
from process_automation.domain.models import ProcessMetrics, StandardProcess

standard = StandardProcess.from_payload(rag_update["standard_profile"])
measured = ProcessMetrics.from_payload(diagnostic_update["actual_metrics"])

gaps = GapAnalyzer().analyse(standard, measured)
for gap in gaps:
    print(f"[{gap.severity.value:>8}] {gap.title}")
    print(f"           {gap.evidence}\n")

[critical] End-to-end duration breaches the documented SLA
           SOP target is 2h but the measured average is 42h (21.0x the SLA, +40h).

[    high] Bottleneck at stage 'Direct manager approval'
           'Direct manager approval' consumes 34.5h (82% of the total cycle time), of which 95% is pure waiting. SOP target for this step is 0.5h.

[  medium] Bottleneck at stage 'Firewall rule provisioning'
           'Firewall rule provisioning' consumes 4.8h (11% of the total cycle time), of which 55% is pure waiting. SOP target for this step is 0.8h.

[  medium] Rework rate above the acceptable threshold
           18 of 140 tickets were reopened (13%); the accepted ceiling is 10%.

[  medium] Recurring operational error: Wrong remote-access profile selected at intake
           Reported repeatedly during 2024-Q4 (140 tickets).

[  medium] Recurring operational error: Employee ID missing, ticket bounced back to HR
           Reported repeatedly during 2024-Q4 (140 tickets).

[  medium]

### ۷.۲ Remediation — a catalogue of strategies, not an `if/elif` chain

Each remedy is a small strategy that declares *which* kind of manual step it
can automate. The engineer asks the registry; it never hard-codes a process
name. Two of them are the heart of the second scenario:

* `ControlRemovalStrategy` — the highest-priority remedy for a review step. It
  optimises purely for lead time and, on the first pass, does not know the
  organisation's compliance limits. **That is exactly the situation the
  Human-in-the-Loop node exists to catch.**
* `SecurityFastTrackStrategy` — keeps the sign-off and removes the *waiting*.
  It is selected automatically once the reviewer protects the control, because
  `ControlRemovalStrategy.applies()` then returns `False` and removes itself
  from the candidate list.

In [18]:
from process_automation.domain.remediation import (
    ControlRemovalStrategy,
    MessengerApprovalStrategy,
    RemediationRegistry,
    SecurityFastTrackStrategy,
)

print(f"{'priority':>9}  {'identifier':<26} handles")
for strategy in sorted(RemediationRegistry().strategies, key=lambda s: -s.priority):
    print(f"{strategy.priority:>9}  {strategy.identifier:<26} {type(strategy).__name__}")

 priority  identifier                 handles
       90  control-removal            ControlRemovalStrategy
       60  messenger-approval-bot     MessengerApprovalStrategy
       55  provisioning-api           ProvisioningApiStrategy
       50  security-fast-track        SecurityFastTrackStrategy
       40  system-to-system-sync      DataEntrySyncStrategy
       35  auto-triage                QueueTriageStrategy
       30  intake-validation          IntakeValidationStrategy
        1  generic-streamlining       GenericStreamliningStrategy


In [19]:
show_source(
    MessengerApprovalStrategy,
    "The remedy the brief expects for the VPN scenario: keep the human "
    "*decision*, delete the human *waiting*.",
)

##### `MessengerApprovalStrategy` — `src/process_automation/domain/remediation.py`

The remedy the brief expects for the VPN scenario: keep the human *decision*, delete the human *waiting*.

```python
class MessengerApprovalStrategy(RemediationStrategy):
    """Keep the human decision, delete the human *waiting*.

    The approver is pushed an actionable card in the corporate messenger the
    moment the request is filed, and the downstream system API is called
    automatically as soon as the button is pressed. This is the remedy the
    project brief expects for the VPN scenario.
    """

    identifier = "messenger-approval-bot"
    priority = 60

    def applies(self, context: RemediationContext) -> bool:
        return context.automation_class is AutomationClass.MANUAL_APPROVAL

    def build(self, context: RemediationContext) -> AutomationProposal:
        step = context.step
        assert step is not None  # guaranteed by ``applies``
        # We keep the SOP target for the step and assume the bot can reach it;
        # everything above the target is the saving.
        gain = max(0.0, context.stage_hours - max(context.target_hours, 0.25))
        return AutomationProposal(
            identifier=self.identifier,
            title=f"Push-based approval bot for '{step.title}'",
            rationale=(
                f"'{step.title}' currently takes {context.stage_hours:g}h, almost all of "
                f"it queue time while the request waits in the approver's inbox. "
                f"The approval *decision* stays with {step.owner}; only the way the "
                f"request reaches them and the way the result is applied become "
                f"automatic."
            ),
            expected_gain_hours=gain,
            implementation_notes=(
                "On ticket creation, post an interactive approve/reject card to the "
                f"corporate messenger account of {step.owner}.",
                "Start an SLA timer; auto-escalate to the line manager's deputy when "
                f"no answer arrives within {max(context.target_hours, 1):g}h.",
                "On approval, call the downstream provisioning API directly instead of "
                "creating a second manual task.",
                "Write the approver identity, timestamp and payload to the audit log so "
                "the control remains fully traceable.",
            ),
            preserved_controls=(step.title,),
            code=_MESSENGER_APPROVAL_CODE,
        )

```

In [20]:
show_source(
    ControlRemovalStrategy,
    "The deliberately aggressive remedy that the systems manager will veto in "
    "scenario 2. Note how `applies()` disables it once the control is protected.",
)

##### `ControlRemovalStrategy` — `src/process_automation/domain/remediation.py`

The deliberately aggressive remedy that the systems manager will veto in scenario 2. Note how `applies()` disables it once the control is protected.

```python
class ControlRemovalStrategy(RemediationStrategy):
    """Pure cycle-time optimisation: delete the sign-off altogether.

    This is intentionally the *highest priority* remedy for a review step: the
    engineer agent optimises for lead time and, on its first pass, does not
    know the organisation's infrastructure and compliance limits. That is
    precisely the situation the Human-in-the-Loop node exists for - the
    reviewer rejects it and the next revision picks a control-preserving
    strategy instead (see the second test scenario of the brief).
    """

    identifier = "control-removal"
    priority = 90

    def applies(self, context: RemediationContext) -> bool:
        if context.automation_class is not AutomationClass.SECURITY_REVIEW:
            return False
        # Once the reviewer protects this control, the strategy removes itself
        # from the candidate list - no special-casing needed in the agent.
        label = context.step.title if context.step else (context.gap.stage or "")
        return not context.is_protected(label)

    def build(self, context: RemediationContext) -> AutomationProposal:
        step = context.step
        assert step is not None
        return AutomationProposal(
            identifier=self.identifier,
            title=f"Replace '{step.title}' with an automated policy check",
            rationale=(
                f"'{step.title}' costs {context.stage_hours:g}h per request and is the "
                f"single largest contributor to the SLA breach. Encoding the review "
                f"criteria as machine-readable rules would let the request be granted "
                f"without waiting for {step.owner}."
            ),
            expected_gain_hours=max(0.0, context.stage_hours - 0.25),
            implementation_notes=(
                "Translate the review checklist into policy-as-code rules.",
                "Grant access automatically when every rule passes.",
                f"Retire the manual '{step.title}' queue.",
            ),
            removed_controls=(step.title,),
            code=_POLICY_AS_CODE,
        )

```

In [21]:
from process_automation.agents import ProcessEngineerAgent

show_source(ProcessEngineerAgent)

##### `ProcessEngineerAgent` — `src/process_automation/agents/process_engineer.py`

```python
class ProcessEngineerAgent(AgentNode):
    """Produces - and, after feedback, re-produces - the Action Plan.

    The agent orchestrates three injected collaborators and owns none of the
    logic itself:

    * :class:`~process_automation.domain.gap_analysis.GapAnalyzer` finds the
      deviations;
    * a :class:`~process_automation.interfaces.FeedbackInterpreter` turns the
      reviewer's sentence into machine-checkable constraints;
    * a :class:`~process_automation.interfaces.SolutionComposer` writes the plan.

    That is why the second scenario of the brief needs no special code path:
    revision 2 is the *same* call with a longer constraint list.
    """

    name = "process_engineer"

    def __init__(
        self,
        analyzer: GapAnalyzer,
        composer: SolutionComposer,
        feedback_interpreter: FeedbackInterpreter,
    ) -> None:
        super().__init__()
        self._analyzer = analyzer
        self._composer = composer
        self._interpreter = feedback_interpreter

    def run(self, state: ProcessAnalysisState) -> Mapping[str, Any]:
        standard = StandardProcess.from_payload(state.get("standard_profile") or {})
        metrics_payload = state.get("actual_metrics") or {}
        revision = int(state.get("revision", 0)) + 1
        trace: list[str] = []

        if not metrics_payload:
            # Without measurements there is nothing to compare against; say so
            # explicitly instead of producing a plausible-looking plan.
            trace.append(self.log("no metrics available - cannot compare"))
            return {
                "identified_gaps": [],
                "proposed_solution": (
                    f"# Action Plan - {standard.process_name or state['process_name']}\n\n"
                    "The monitoring tools returned no data for this process, so the "
                    "documented standard cannot be compared with reality. "
                    "Onboard the process into the ServiceDesk analytics before "
                    "requesting an automation analysis."
                ),
                "action_plan": {},
                "revision": revision,
                "approved": False,
                "human_feedback": "",
                "trace": trace,
            }

        metrics = ProcessMetrics.from_payload(metrics_payload)

        # 1) Carry forward every constraint stated in previous revisions, then
        #    translate any new feedback into additional ones. Constraints are
        #    cumulative, which is what makes the revision loop converge.
        constraints = tuple(
            Constraint.from_payload(item) for item in state.get("constraints", [])
        )
        feedback = (state.get("human_feedback") or "").strip()
        if feedback:
            new_constraints = self._interpreter.interpret(feedback, standard)
            constraints = merge_constraints(constraints, new_constraints)
            trace.append(self.log(f"re-planning after human feedback: '{feedback}'"))
            for constraint in new_constraints:
                trace.append(self.log(f"  derived constraint -> {constraint.describe()}"))

        # 2) Compare should-be with as-is.
        gaps = self._analyzer.analyse(standard, metrics)
        trace.append(
            self.log(
                f"revision {revision}: {len(gaps)} gap(s) identified "
                f"({', '.join(gap.severity.value for gap in gaps) or 'none'})"
            )
        )
        for gap in gaps:
            trace.append(self.log(f"  [{gap.severity.value}] {gap.title}"))

        # 3) Compose the plan under the current constraints.
        plan = self._composer.compose(standard, metrics, gaps, constraints, revision)
        for proposal in plan.proposals:
            trace.append(
                self.log(
                    f"  proposal '{proposal.title}' "
                    f"(~{proposal.expected_gain_hours:.1f}h, {proposal.gain_kind})"
                )
            )
            if proposal.removed_controls:
                trace.append(
                    self.log(
                        "  ! this proposal removes the control(s): "
                        + ", ".join(proposal.removed_controls)
                    )
                )
        trace.append(
            self.log(
                f"projected duration {plan.baseline_hours:g}h -> "
                f"{plan.projected_hours:.1f}h (SLA {plan.sla_hours:g}h)"
            )
        )

        return {
            "identified_gaps": [gap.to_payload() for gap in gaps],
            "proposed_solution": plan.to_markdown(),
            "action_plan": plan.to_payload(),
            "constraints": [constraint.to_payload() for constraint in constraints],
            "revision": revision,
            # The feedback has now been consumed and turned into constraints;
            # clearing it prevents the router from looping on a stale value.
            "human_feedback": "",
            "approved": False,
            "trace": trace,
        }

```

---
## ۸. گره ۴ — تاییدیه مدیر سیستم‌ها (Human-in-the-Loop)

**نقش:** کنترل و تایید نهایی توسط شما.
**وظیفه:** استفاده از قابلیت Checkpointer در LangGraph. پیش از نهایی شدن
گزارش، سیستم متوقف می‌شود. اگر راهکار منطقی بود تایید می‌کنید، و اگر با
محدودیت‌های زیرساختی شرکت مغایرت داشت، فیدبک می‌دهید تا سیستم راهکار جایگزین
تولید کند.

Two responsibilities are deliberately kept apart:

* **the pause** belongs to the graph runtime — the graph is compiled with
  `interrupt_before=["human_review"]` and a `MemorySaver` checkpointer;
* **the asking** belongs to an `ApprovalGateway`, of which there are several
  interchangeable implementations.

That separation is why the *same compiled graph* serves an interactive console
session, a reproducible scripted run and an automated test, with no change to
any node.

### ۸.۱ The reviewer's feedback becomes a constraint, not a hint

A compliance limit must never depend on an LLM's sampling temperature, so the
interpreter is deterministic and understands both English and Persian.

In [22]:
from process_automation.domain.feedback import RuleBasedFeedbackInterpreter

interpreter = RuleBasedFeedbackInterpreter()
db_standard = knowledge_base.get("SOP-SEC-021").to_standard_process()

for sentence in [
    "The security approval cannot be removed. Propose a way to accelerate the security review instead.",
    "تاییدیه امنیتی غیرقابل حذف است. راهکاری برای تسریع بررسی امنیتی پیشنهاد بده.",
    "Make it cheaper.",
]:
    constraint = interpreter.interpret(sentence, db_standard)[0]
    print(f"feedback   : {sentence}")
    print(f"constraint : {constraint.describe()}\n")

feedback   : The security approval cannot be removed. Propose a way to accelerate the security review instead.
constraint : preserve control -> Security officer review (accelerate the control instead of removing it)

feedback   : تاییدیه امنیتی غیرقابل حذف است. راهکاری برای تسریع بررسی امنیتی پیشنهاد بده.
constraint : preserve control -> Security officer review (accelerate the control instead of removing it)

feedback   : Make it cheaper.
constraint : generic (Make it cheaper.)



In [23]:
from process_automation.agents import HumanReviewNode

show_source(HumanReviewNode)

##### `HumanReviewNode` — `src/process_automation/agents/human_review.py`

```python
class HumanReviewNode(AgentNode):
    """Records the systems manager's verdict on the current Action Plan."""

    name = "human_review"

    def run(self, state: ProcessAnalysisState) -> Mapping[str, Any]:
        feedback = (state.get("human_feedback") or "").strip()
        revision = int(state.get("revision", 1))

        # Feedback always wins over a stale ``approved`` flag: a reviewer who
        # writes a correction has not approved the plan, whatever else is set.
        approved = bool(state.get("approved", False)) and not feedback

        if approved:
            trace = [self.log(f"revision {revision} APPROVED by the systems manager")]
            return {"approved": True, "human_feedback": "", "trace": trace}

        trace = [
            self.log(f"revision {revision} REJECTED - feedback: '{feedback}'")
            if feedback
            else self.log(f"revision {revision} not approved and no feedback given")
        ]
        return {
            "approved": False,
            "human_feedback": feedback,
            "feedback_log": [f"revision {revision}: {feedback}"] if feedback else [],
            "trace": trace,
        }

```

In [24]:
from process_automation.infrastructure.approval import (
    AutoApproveGateway,
    ConsoleApprovalGateway,
    ScriptedApprovalGateway,
)

show_source(
    ScriptedApprovalGateway,
    "Used in section 12 so the grader's run is identical to this one. "
    "`ConsoleApprovalGateway` (section 12.3) asks you interactively instead.",
)

##### `ScriptedApprovalGateway` — `src/process_automation/infrastructure/approval.py`

Used in section 12 so the grader's run is identical to this one. `ConsoleApprovalGateway` (section 12.3) asks you interactively instead.

```python
class ScriptedApprovalGateway:
    """Replays a pre-recorded list of decisions.

    This is what makes the notebook's second scenario reproducible: the
    reviewer's exact sentence is scripted, so the run is identical every time
    it is executed by the grader.
    """

    def __init__(self, decisions: Sequence[ApprovalDecision]) -> None:
        self._queue: deque[ApprovalDecision] = deque(decisions)
        self._history: list[ApprovalDecision] = []

    @property
    def name(self) -> str:
        return "scripted"

    @property
    def history(self) -> tuple[ApprovalDecision, ...]:
        return tuple(self._history)

    def review(self, plan: ActionPlan, *, revision: int) -> ApprovalDecision:
        # Once the script is exhausted the reviewer approves, so a run can
        # never loop forever because of a missing line in the script.
        decision = (
            self._queue.popleft()
            if self._queue
            else ApprovalDecision(approved=True, feedback="", reviewer="scripted")
        )
        self._history.append(decision)
        return decision

```

---
## ۹. مرحله ۳ — پیاده‌سازی یال‌های شرطی (Conditional Edges)

مسیر گراف باید به گونه‌ای تعریف شود که پس از تولید `proposed_solution`، سیستم
متوقف شود. در صورت دریافت `human_feedback`، گراف مجدداً به گره مهندس بازگردد تا
راهکار جدیدی بر اساس فیدبک تدوین شود.

Routing is kept **out of the nodes**: a node answers *"what did I find?"*, a
router answers *"where does the flow go next?"*. The revision loop can then be
re-tuned — or replaced by an escalation to a second reviewer — without
reopening a single agent.

`max_revisions` is a safety valve: without it, a reviewer who keeps rejecting
would spin the graph forever.

In [25]:
from process_automation.graph import WorkflowGraphBuilder, build_review_router

show_source(build_review_router)

##### `build_review_router` — `src/process_automation/graph/routing.py`

```python
def build_review_router(max_revisions: int = 3) -> Callable[[ProcessAnalysisState], str]:
    """Return the router used after the Human-in-the-Loop node.

    The decision table:

    ======================================  ==========================
    state                                   next
    ======================================  ==========================
    approved                                END
    feedback given, revisions left          back to ``process_engineer``
    feedback given, revision budget spent   END (with a warning)
    no feedback and not approved            END (reviewer abstained)
    ======================================  ==========================

    ``max_revisions`` is a safety valve: without it, a reviewer who keeps
    rejecting would spin the graph forever.
    """

    def route_after_review(state: ProcessAnalysisState) -> str:
        if state.get("approved"):
            return FINISH
        feedback = (state.get("human_feedback") or "").strip()
        if not feedback:
            # Neither approval nor feedback: nothing actionable to re-plan.
            return FINISH
        if int(state.get("revision", 0)) >= max_revisions:
            return FINISH
        return REVISE

    return route_after_review

```

In [26]:
show_source(WorkflowGraphBuilder)

##### `WorkflowGraphBuilder` — `src/process_automation/graph/builder.py`

```python
class WorkflowGraphBuilder:
    """Builds and compiles the multi-agent workflow graph."""

    def __init__(
        self,
        documentation_agent: AgentNode,
        diagnostic_agent: AgentNode,
        process_engineer: AgentNode,
        human_review: AgentNode,
        *,
        max_revisions: int = 3,
    ) -> None:
        self._documentation = documentation_agent
        self._diagnostic = diagnostic_agent
        self._engineer = process_engineer
        self._human_review = human_review
        self._max_revisions = max_revisions

    @property
    def node_names(self) -> tuple[str, ...]:
        return (
            self._documentation.name,
            self._diagnostic.name,
            self._engineer.name,
            self._human_review.name,
        )

    def build(self) -> StateGraph:
        """Declare nodes and edges (no compilation, no checkpointer yet)."""
        graph = StateGraph(ProcessAnalysisState)

        # -- nodes: one per agent, registered under the agent's own name -----
        graph.add_node(self._documentation.name, self._documentation)
        graph.add_node(self._diagnostic.name, self._diagnostic)
        graph.add_node(self._engineer.name, self._engineer)
        graph.add_node(self._human_review.name, self._human_review)

        # -- linear part: understand the standard, then measure reality ------
        graph.add_edge(START, self._documentation.name)
        graph.add_edge(self._documentation.name, self._diagnostic.name)
        graph.add_edge(self._diagnostic.name, self._engineer.name)

        # -- the plan always goes to the reviewer; the interrupt is applied
        #    at compile time, right before this node executes ----------------
        graph.add_edge(self._engineer.name, self._human_review.name)

        # -- conditional edge: approve and stop, or re-plan with feedback ----
        graph.add_conditional_edges(
            self._human_review.name,
            build_review_router(self._max_revisions),
            review_route_map(self._engineer.name),
        )
        return graph

    def compile(
        self,
        checkpointer: Any | None = None,
        *,
        interrupt_before: Sequence[str] | None = None,
    ) -> Any:
        """Compile the graph with a checkpointer and the human interrupt.

        The checkpointer is what makes the Human-in-the-Loop possible: it
        persists the state at the interrupt so the run can be resumed - even
        minutes later, or from a different process - once the systems manager
        has answered.
        """
        return self.build().compile(
            checkpointer=checkpointer if checkpointer is not None else MemorySaver(),
            interrupt_before=list(
                interrupt_before if interrupt_before is not None else [self._human_review.name]
            ),
        )

```

---
## ۱۰. ساخت سیستم و نمایش گراف

### ۱۰.۱ Composition root

Every module in this project depends on **abstractions**. `build_system()` is
the single place where those abstractions are bound to concrete classes — the
practical pay-off of the Dependency Inversion Principle: to run the whole
system against FAISS, a real Jira API and a hosted LLM you edit *one file*.

In [27]:
from process_automation.application.container import build_system

show_source(build_system)

##### `build_system` — `src/process_automation/application/container.py`

```python
def build_system(
    settings: Settings | None = None,
    *,
    approval_gateway: ApprovalGateway | None = None,
    remediation_registry: RemediationRegistry | None = None,
) -> MultiAgentSystem:
    """Wire the whole system together and return it ready to run."""
    settings = settings or Settings()

    # -- Node 1 dependencies: knowledge base -> embeddings -> vector store --
    knowledge_base = MarkdownKnowledgeBaseLoader(
        settings.retrieval.knowledge_base_dir
    ).load()
    embeddings = HashingTfidfEmbeddings(settings.retrieval.embedding_dimensions)
    vector_store = InMemoryVectorStore.from_knowledge_base(
        knowledge_base, embeddings, min_score=settings.retrieval.min_score
    )
    documentation_agent = DocumentationAgent(
        retriever=vector_store,
        parser=StructuredSopParser(knowledge_base),
        top_k=settings.retrieval.top_k,
    )

    # -- Node 2 dependencies: the mock ServiceDesk tools --------------------
    dataset = ServiceDeskDataset.from_json(settings.diagnostics.dataset_path)
    tools = ServiceDeskToolFactory(dataset).build()
    metrics_provider = ServiceDeskMetricsProvider(tools)
    diagnostic_agent = DiagnosticAgent(metrics_provider)

    # -- Node 3 dependencies: analysis + composition ------------------------
    composer: Any = RuleBasedSolutionComposer(
        remediation_registry or RemediationRegistry(),
        max_proposals=settings.engineering.max_proposals,
        allow_control_removal=settings.engineering.allow_control_removal,
    )
    language_model = build_language_model(settings)
    language_model_name = "none (deterministic mode)"
    if language_model is not None:
        # Decorator, not replacement: the numbers stay deterministic and only
        # the management summary is written by the model.
        composer = LlmNarrativeComposer(composer, language_model)
        language_model_name = language_model.name
        _logger.info("language model enabled for the plan narrative: %s", language_model_name)

    process_engineer = ProcessEngineerAgent(
        analyzer=GapAnalyzer(),
        composer=composer,
        feedback_interpreter=RuleBasedFeedbackInterpreter(),
    )

    # -- Node 4 and the graph ----------------------------------------------
    human_review = HumanReviewNode()
    graph_builder = WorkflowGraphBuilder(
        documentation_agent,
        diagnostic_agent,
        process_engineer,
        human_review,
        max_revisions=settings.review.max_revisions,
    )
    compiled_graph = graph_builder.compile()

    session = WorkflowSession(
        compiled_graph,
        approval_gateway or AutoApproveGateway(),
        human_node=human_review.name,
        default_thread_id=settings.review.default_thread_id,
        max_interrupts=settings.review.max_revisions + 2,
    )

    return MultiAgentSystem(
        settings=settings,
        knowledge_base=knowledge_base,
        vector_store=vector_store,
        tools=tools,
        metrics_provider=metrics_provider,
        graph_builder=graph_builder,
        compiled_graph=compiled_graph,
        session=session,
        visualizer=MermaidGraphVisualizer(),
        language_model_name=language_model_name,
    )

```

In [28]:
from process_automation import Settings

system = build_system(Settings(), approval_gateway=AutoApproveGateway())
print(system.describe())

Multi-agent process analysis system
  knowledge base : 3 SOP document(s), 15 indexed chunk(s)
  tools          : fetch_process_metrics, fetch_stage_timings
  graph nodes    : documentation_agent -> diagnostic_agent -> process_engineer -> human_review
  reasoning      : none (deterministic mode)
  max revisions  : 3


### ۱۰.۲ نمایش گراف تولید شده — `.get_graph().draw_mermaid_png()`

The brief asks for exactly this call. It renders through the remote
`mermaid.ink` service, so on an offline or firewalled machine it cannot
succeed — `MermaidGraphVisualizer` therefore tries the PNG first and falls back
to writing the Mermaid source, which is rendered natively by the next cell.
The deliverable is inspectable either way.

In [29]:
from IPython.display import Image

rendering = system.visualizer.render(system.compiled_graph)
print(rendering["note"])
print("written to:", rendering["path"])

if rendering["format"] == "png":
    display(Image(rendering["png_bytes"]))

draw_mermaid_png() needs the mermaid.ink renderer, which is unreachable here (ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`). The Mermaid source was written instead - paste it into any Mermaid viewer, or re-run this cell on a machine with internet access.
written to: artifacts/workflow_graph.mmd


In [30]:
# The Mermaid source of the compiled graph - always available.
# Note `human_review`, which carries the `__interrupt = before` marker: that is
# the checkpointer pause required by node 4 of the brief.
print(rendering["source"])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	documentation_agent(documentation_agent)
	diagnostic_agent(diagnostic_agent)
	process_engineer(process_engineer)
	human_review(human_review<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> documentation_agent;
	diagnostic_agent --> process_engineer;
	documentation_agent --> diagnostic_agent;
	human_review -. &nbsp;finish&nbsp; .-> __end__;
	human_review -. &nbsp;revise&nbsp; .-> process_engineer;
	process_engineer --> human_review;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [31]:
# Rendered natively by the notebook front-end (GitHub, VS Code, JupyterLab).
display(Markdown("```mermaid\n" + rendering["source"] + "\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	documentation_agent(documentation_agent)
	diagnostic_agent(diagnostic_agent)
	process_engineer(process_engineer)
	human_review(human_review<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> documentation_agent;
	diagnostic_agent --> process_engineer;
	documentation_agent --> diagnostic_agent;
	human_review -. &nbsp;finish&nbsp; .-> __end__;
	human_review -. &nbsp;revise&nbsp; .-> process_engineer;
	process_engineer --> human_review;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

---
## ۱۱. سناریوی اول — شناسایی گلوگاه و پیشنهاد اتوماسیون موفق

> **ورودی:** بررسی فرآیند تخصیص دسترسی VPN به پرسنل جدید
>
> **جریان مورد انتظار:**
> * گره RAG: طبق مستندات، این کار باید در **۲ ساعت** انجام شود.
> * گره Tools: دیتای فرضی نشان می‌دهد میانگین زمان انجام **۴۲ ساعت** است و
>   بیشترین توقف در مرحله «تاییدیه مدیر مستقیم» رخ می‌دهد.
> * گره مهندس پیشنهاد می‌دهد: «یک اسکریپت پایتون نوشته شود که به محض ثبت
>   درخواست، تاییدیه را از طریق پیام‌رسان سازمانی از مدیر بگیرد و API فایروال
>   را مستقیماً فراخوانی کند.»

The log below is the live execution trace: every RAG hit with its similarity
score, every tool call, every gap, and the moment the graph stops for the
human.

In [32]:
rule("SCENARIO 1 — VPN access provisioning for new personnel")

scenario_1 = system.session.run(
    "Review the VPN access provisioning process for new personnel",
    thread_id="scenario-1",
)


SCENARIO 1 — VPN access provisioning for new personnel
starting analysis of 'Review the VPN access provisioning process for new personnel' on thread 'scenario-1'


RAG lookup for: 'Review the VPN access provisioning process for new personnel' (top_k=4)


  hit score=0.398 doc=SOP-IT-014 section='header'


  hit score=0.329 doc=SOP-IT-014 section='2. Service level agreement'


  hit score=0.240 doc=SOP-IT-014 section='4. Policy notes'


...

  proposal 'Validate the request at intake instead of after execution' (~5.4h, amortised)


projected duration 42h -> 4.0h (SLA 2h)


graph interrupted before 'human_review' (revision 1) - waiting for the human


reviewer decision: APPROVED


revision 1 APPROVED by the systems manager


In [33]:
print(f"approved   : {scenario_1.approved}")
print(f"revisions  : {scenario_1.revisions}")
print(f"interrupts : {scenario_1.interrupts}  {scenario_1.checkpoints}")
print(f"outcome    : {scenario_1.stopped_reason}")

approved   : True
revisions  : 1
interrupts : 1  ("interrupt #1 before 'human_review'",)
outcome    : completed


### ۱۱.۱ خروجی گره ۱ — the documented standard extracted by RAG

In [34]:
print(scenario_1.state["standard_sop"])
print("\nretrieved chunks that produced it:")
for chunk in scenario_1.state["retrieved_documents"]:
    print(f"  {chunk['score']:.3f}  {chunk['document_id']:<12} §{chunk['section']}")

Process: VPN access provisioning for new personnel
Documented SLA: 2 hour(s) end-to-end
Approved steps:
  1. Request registration - owner: ServiceDesk intake bot - target: 0.1h - type: automated
  2. Direct manager approval - owner: Direct Manager - target: 0.5h - type: manual_approval
  3. Security policy check - owner: Security Operations rule engine - target: 0.4h - type: automated
  4. Firewall rule provisioning - owner: Network Operations - target: 0.8h - type: manual_provisioning
  5. User notification and handover - owner: ServiceDesk - target: 0.2h - type: automated
Policy notes:
  - The direct manager's approval records business need; it may be delegated to a named deputy but never skipped.
  - Every granted profile must be time-boxed and re-certified every 180 days.
  - Standard remote-access profiles are pre-approved by Information Security, so no individual security sign-off is required for this process.
  - All access decisions are written to the immutable audit log.
Sourc

### ۱۱.۲ خروجی گره ۲ — the reality measured through the tools

In [35]:
import json

print(json.dumps(scenario_1.state["actual_metrics"], indent=2, ensure_ascii=False))

{
  "process_name": "VPN access provisioning for new personnel",
  "average_resolution_hours": 42.0,
  "ticket_volume": 140,
  "reopened_tickets": 18,
  "reopen_rate": 0.1286,
  "common_errors": [
    "Wrong remote-access profile selected at intake",
    "Employee ID missing, ticket bounced back to HR",
    "Firewall rule typed against the wrong user group"
  ],
  "stage_timings": [
    {
      "stage": "Request registration",
      "average_hours": 0.4,
      "waiting_share": 0.1
    },
    {
      "stage": "Direct manager approval",
      "average_hours": 34.5,
      "waiting_share": 0.95
    },
    {
      "stage": "Security policy check",
      "average_hours": 1.6,
      "waiting_share": 0.3
    },
    {
      "stage": "Firewall rule provisioning",
      "average_hours": 4.8,
      "waiting_share": 0.55
    },
    {
      "stage": "User notification and handover",
      "average_hours": 0.7,
      "waiting_share": 0.2
    }
  ],
  "sample_period": "2024-Q4 (140 tickets)"
}


### ۱۱.۳ خروجی گره ۳ — the Action Plan

Note the arithmetic: **42 h → 4.0 h**. The plan does not claim to reach the 2 h
SLA, and says so; an honest projection is worth more to a systems manager than
an optimistic one.

In [36]:
display(Markdown(scenario_1.state["proposed_solution"]))

# Action Plan - VPN access provisioning for new personnel (revision 1)

The documented SLA for 'VPN access provisioning for new personnel' is 2h, but the 2024-Q4 (140 tickets) sample shows an average of 42h - 21.0x the standard. The dominant bottleneck is 'Direct manager approval' with 34.5h (82% of the cycle, 95% of it pure waiting). 7 gap(s) were identified and 3 automation measure(s) are proposed below.

## 1. Identified gaps
- **End-to-end duration breaches the documented SLA** (critical)
  - evidence: SOP target is 2h but the measured average is 42h (21.0x the SLA, +40h).
- **Bottleneck at stage 'Direct manager approval'** (high) [stage: Direct manager approval]
  - evidence: 'Direct manager approval' consumes 34.5h (82% of the total cycle time), of which 95% is pure waiting. SOP target for this step is 0.5h.
- **Bottleneck at stage 'Firewall rule provisioning'** (medium) [stage: Firewall rule provisioning]
  - evidence: 'Firewall rule provisioning' consumes 4.8h (11% of the total cycle time), of which 55% is pure waiting. SOP target for this step is 0.8h.
- **Rework rate above the acceptable threshold** (medium)
  - evidence: 18 of 140 tickets were reopened (13%); the accepted ceiling is 10%.
- **Recurring operational error: Wrong remote-access profile selected at intake** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (140 tickets).
- **Recurring operational error: Employee ID missing, ticket bounced back to HR** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (140 tickets).
- **Recurring operational error: Firewall rule typed against the wrong user group** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (140 tickets).

## 2. Proposed automation
### Push-based approval bot for 'Direct manager approval'
'Direct manager approval' currently takes 34.5h, almost all of it queue time while the request waits in the approver's inbox. The approval *decision* stays with Direct Manager; only the way the request reaches them and the way the result is applied become automatic.

*Expected saving:* ~34 hour(s) per request.
- On ticket creation, post an interactive approve/reject card to the corporate messenger account of Direct Manager.
- Start an SLA timer; auto-escalate to the line manager's deputy when no answer arrives within 1h.
- On approval, call the downstream provisioning API directly instead of creating a second manual task.
- Write the approver identity, timestamp and payload to the audit log so the control remains fully traceable.
- Controls kept intact: Direct manager approval

```python
"""Push-based approval: ask the manager in the messenger, provision on YES."""

import datetime as dt
import requests

MESSENGER_API = "https://messenger.corp.local/api/v1"
SERVICE_DESK_API = "https://servicedesk.corp.local/api/v2"
ESCALATION_AFTER = dt.timedelta(hours=1)


def request_approval(ticket: dict) -> dict:
    """Post an interactive approve/reject card to the approver."""
    response = requests.post(
        f"{MESSENGER_API}/cards",
        json={
            "to": ticket["approver_id"],
            "title": f"Access request {ticket['id']} - {ticket['requester']}",
            "body": ticket["justification"],
            "actions": [
                {"id": "approve", "label": "Approve", "style": "primary"},
                {"id": "reject", "label": "Reject", "style": "danger"},
            ],
            "callback_url": f"{SERVICE_DESK_API}/tickets/{ticket['id']}/decision",
            "expires_at": (dt.datetime.utcnow() + ESCALATION_AFTER).isoformat(),
        },
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


def on_decision(ticket: dict, decision: dict) -> None:
    """Callback invoked by the messenger when the approver clicks a button."""
    audit(ticket, decision)                       # the control stays traceable
    if decision["action"] != "approve":
        close_ticket(ticket, status="rejected", note=decision.get("comment", ""))
        return
    provision(ticket)                             # call the target system API
    close_ticket(ticket, status="done")


def escalate_if_silent(ticket: dict) -> None:
    """Re-route to the deputy when the SLA timer expires (no lost requests)."""
    if ticket["status"] == "awaiting_approval" and _age(ticket) > ESCALATION_AFTER:
        ticket["approver_id"] = ticket["deputy_approver_id"]
        request_approval(ticket)
```
### Direct API provisioning for 'Firewall rule provisioning'
'Firewall rule provisioning' is executed by hand by Network Operations, which costs 4.8h and is the source of the recurring configuration errors visible in the ticket data.

*Expected saving:* ~4 hour(s) per request.
- Model the target configuration as a versioned template.
- Apply it through the vendor API inside a transaction, with an automatic rollback when validation fails.
- Re-read the applied configuration and attach the diff to the ticket.

```python
"""Provision the target system from a versioned template instead of by hand."""

import requests

FIREWALL_API = "https://firewall.corp.local/api/v3"


def provision(ticket: dict) -> dict:
    """Apply the access template, verify it, and attach the diff to the ticket."""
    payload = {
        "user": ticket["requester_uid"],
        "profile": ticket["access_profile"],       # template, not free-form rules
        "valid_until": ticket["valid_until"],      # every grant is time-boxed
        "ticket": ticket["id"],
    }
    applied = requests.post(f"{FIREWALL_API}/vpn-users", json=payload, timeout=15)
    applied.raise_for_status()

    # Read back and compare: configuration drift is detected immediately.
    current = requests.get(
        f"{FIREWALL_API}/vpn-users/{ticket['requester_uid']}", timeout=15
    ).json()
    if current["profile"] != payload["profile"]:
        rollback(ticket)
        raise RuntimeError(f"provisioning drift for ticket {ticket['id']}")
    return current
```
### Validate the request at intake instead of after execution
Rework is driven by defects that are only discovered downstream (Wrong remote-access profile selected at intake, Employee ID missing, ticket bounced back to HR, Firewall rule typed against the wrong user group). Each reopened ticket re-runs the whole process.

*Expected saving:* ~5.4 hour(s) per request.
- Make the intake form dynamic: required fields depend on the request type, and free text is replaced by pick-lists wherever possible.
- Validate against the source of truth (HR system, CMDB) before the ticket can be submitted.
- Block submission - rather than the whole downstream process - when a prerequisite is missing.

## 4. Expected effect
- Cycle-time saving: ~38.0 h per request.
- Additional amortised saving (avoided rework): ~5.4 h per request on average.
- Projected end-to-end duration: **42.0 h -> 4.0 h** (still above the 2 h SLA).

### ۱۱.۴ لاگ کامل اجرا — the full audit trail stored in the state

In [37]:
scenario_1.print_trace()

[documentation_agent] RAG lookup for: 'Review the VPN access provisioning process for new personnel' (top_k=4)
[documentation_agent]   hit score=0.398 doc=SOP-IT-014 section='header'
[documentation_agent]   hit score=0.329 doc=SOP-IT-014 section='2. Service level agreement'
[documentation_agent]   hit score=0.240 doc=SOP-IT-014 section='4. Policy notes'
[documentation_agent]   hit score=0.229 doc=SOP-IT-014 section='1. Purpose and scope'
[documentation_agent] extracted SLA=2h and 5 approved step(s) from SOP-IT-014
[diagnostic_agent] calling tool(s) fetch_process_metrics, fetch_stage_timings for process 'VPN access provisioning for new personnel'
[diagnostic_agent] measured average=42h over 140 ticket(s) [2024-Q4 (140 tickets)], reopened=18 (13%)
[diagnostic_agent]   stage 'Request registration': 0.4h (1% of cycle, 10% waiting)
[diagnostic_agent]   stage 'Direct manager approval': 34.5h (82% of cycle, 95% waiting)
[diagnostic_agent]   stage 'Security policy check': 1.6h (4% of cycle, 30

---
## ۱۲. سناریوی دوم — مداخله انسانی و اصلاح مسیر

> **ورودی:** پیشنهاد اتوماسیون گره مهندس برای دسترسی‌های حساس دیتابیس.
>
> **جریان مورد انتظار:**
> * گره مهندس پیشنهاد می‌دهد که تاییدیه امنیتی حذف و خودکار شود.
> * گراف متوقف می‌شود. شما (Human-in-the-Loop) فیدبک می‌دهید: «تاییدیه امنیتی
>   غیرقابل حذف است. راهکاری برای تسریع بررسی امنیتی پیشنهاد بده.»
> * سیستم مجدداً فعال شده و پیشنهاد جایگزین (مثلاً تولید یک داشبورد سریع برای
>   افسر امنیت) را ارائه می‌دهد.

### ۱۲.۱ Revision 1 — the graph stops before the plan can be executed

First, let us watch the interrupt itself. The graph is invoked with **no
approval gateway attached**, so it runs until the checkpointer stops it — and
then simply waits.

In [38]:
rule("SCENARIO 2 — revision 1: the graph pauses and waits for the human")

config = {"configurable": {"thread_id": "scenario-2"}}
paused = system.compiled_graph.invoke(
    initial_state("privileged access to a production database"), config
)

snapshot = system.compiled_graph.get_state(config)
print(f"\ngraph is paused. next node = {snapshot.next}")
print(f"checkpoint id = {snapshot.config['configurable']['checkpoint_id']}")


SCENARIO 2 — revision 1: the graph pauses and waits for the human
RAG lookup for: 'privileged access to a production database' (top_k=4)


  hit score=0.481 doc=SOP-SEC-021 section='header'


  hit score=0.355 doc=SOP-SEC-021 section='1. Purpose and scope'


  hit score=0.210 doc=SOP-SEC-021 section='2. Service level agreement'


  hit score=0.179 doc=SOP-SEC-021 section='4. Policy notes'


...

  ! this proposal removes the control(s): Security officer review


  proposal 'Push-based approval bot for 'Direct manager approval'' (~2.5h, cycle_time)


  proposal 'Direct API provisioning for 'DBA privilege grant'' (~1.2h, cycle_time)


projected duration 26h -> 4.6h (SLA 8h)



graph is paused. next node = ('human_review',)
checkpoint id = 1f1b7283-8653-6430-8003-2a8452cdadf6


In [39]:
# This is what the engineer wants to do - and why the reviewer must see it
# before anything is executed.
for proposal in paused["action_plan"]["proposals"]:
    flag = "⚠ REMOVES A CONTROL" if proposal["removed_controls"] else "✓ keeps controls"
    print(f"{flag:<22} {proposal['title']}  (~{proposal['expected_gain_hours']}h)")
    if proposal["removed_controls"]:
        print(f"{'':<22} would delete: {', '.join(proposal['removed_controls'])}")

⚠ REMOVES A CONTROL    Replace 'Security officer review' with an automated policy check  (~17.75h)
                       would delete: Security officer review
✓ keeps controls       Push-based approval bot for 'Direct manager approval'  (~2.5h)
✓ keeps controls       Direct API provisioning for 'DBA privilege grant'  (~1.2h)


In [40]:
display(Markdown(paused["proposed_solution"]))

# Action Plan - Privileged database access provisioning (revision 1)

The documented SLA for 'Privileged database access provisioning' is 8h, but the 2024-Q4 (85 tickets) sample shows an average of 26h - 3.2x the standard. The dominant bottleneck is 'Security officer review' with 18h (69% of the cycle, 80% of it pure waiting). 8 gap(s) were identified and 3 automation measure(s) are proposed below.

## 1. Identified gaps
- **End-to-end duration breaches the documented SLA** (high)
  - evidence: SOP target is 8h but the measured average is 26h (3.2x the SLA, +18h).
- **Bottleneck at stage 'Security officer review'** (high) [stage: Security officer review]
  - evidence: 'Security officer review' consumes 18h (69% of the total cycle time), of which 80% is pure waiting. SOP target for this step is 4h.
- **Bottleneck at stage 'Direct manager approval'** (medium) [stage: Direct manager approval]
  - evidence: 'Direct manager approval' consumes 3.5h (13% of the total cycle time), of which 80% is pure waiting. SOP target for this step is 1h.
- **Bottleneck at stage 'DBA privilege grant'** (medium) [stage: DBA privilege grant]
  - evidence: 'DBA privilege grant' consumes 3.2h (12% of the total cycle time), of which 45% is pure waiting. SOP target for this step is 2h.
- **Rework rate above the acceptable threshold** (medium)
  - evidence: 9 of 85 tickets were reopened (11%); the accepted ceiling is 10%.
- **Recurring operational error: Business justification too vague, returned by the security officer** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (85 tickets).
- **Recurring operational error: Requested privilege level higher than the peer group baseline** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (85 tickets).
- **Recurring operational error: Expiry date left empty, grant defaulted to permanent** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (85 tickets).

## 2. Proposed automation
### Replace 'Security officer review' with an automated policy check
'Security officer review' costs 18h per request and is the single largest contributor to the SLA breach. Encoding the review criteria as machine-readable rules would let the request be granted without waiting for Information Security Officer.

*Expected saving:* ~17.75 hour(s) per request.
- Translate the review checklist into policy-as-code rules.
- Grant access automatically when every rule passes.
- Retire the manual 'Security officer review' queue.

```python
"""Encode the review checklist as rules and grant access automatically."""

RULES = (
    lambda request: request["privilege_level"] <= 2,
    lambda request: request["data_classification"] <= 2,
    lambda request: request["peer_group_match"] >= 0.8,
    lambda request: request["past_incidents"] == 0,
)


def auto_decide(request: dict) -> bool:
    """Grant when every rule passes - no human queue involved."""
    return all(rule(request) for rule in RULES)
```
### Push-based approval bot for 'Direct manager approval'
'Direct manager approval' currently takes 3.5h, almost all of it queue time while the request waits in the approver's inbox. The approval *decision* stays with Direct Manager; only the way the request reaches them and the way the result is applied become automatic.

*Expected saving:* ~2.5 hour(s) per request.
- On ticket creation, post an interactive approve/reject card to the corporate messenger account of Direct Manager.
- Start an SLA timer; auto-escalate to the line manager's deputy when no answer arrives within 1h.
- On approval, call the downstream provisioning API directly instead of creating a second manual task.
- Write the approver identity, timestamp and payload to the audit log so the control remains fully traceable.
- Controls kept intact: Direct manager approval

```python
"""Push-based approval: ask the manager in the messenger, provision on YES."""

import datetime as dt
import requests

MESSENGER_API = "https://messenger.corp.local/api/v1"
SERVICE_DESK_API = "https://servicedesk.corp.local/api/v2"
ESCALATION_AFTER = dt.timedelta(hours=1)


def request_approval(ticket: dict) -> dict:
    """Post an interactive approve/reject card to the approver."""
    response = requests.post(
        f"{MESSENGER_API}/cards",
        json={
            "to": ticket["approver_id"],
            "title": f"Access request {ticket['id']} - {ticket['requester']}",
            "body": ticket["justification"],
            "actions": [
                {"id": "approve", "label": "Approve", "style": "primary"},
                {"id": "reject", "label": "Reject", "style": "danger"},
            ],
            "callback_url": f"{SERVICE_DESK_API}/tickets/{ticket['id']}/decision",
            "expires_at": (dt.datetime.utcnow() + ESCALATION_AFTER).isoformat(),
        },
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


def on_decision(ticket: dict, decision: dict) -> None:
    """Callback invoked by the messenger when the approver clicks a button."""
    audit(ticket, decision)                       # the control stays traceable
    if decision["action"] != "approve":
        close_ticket(ticket, status="rejected", note=decision.get("comment", ""))
        return
    provision(ticket)                             # call the target system API
    close_ticket(ticket, status="done")


def escalate_if_silent(ticket: dict) -> None:
    """Re-route to the deputy when the SLA timer expires (no lost requests)."""
    if ticket["status"] == "awaiting_approval" and _age(ticket) > ESCALATION_AFTER:
        ticket["approver_id"] = ticket["deputy_approver_id"]
        request_approval(ticket)
```
### Direct API provisioning for 'DBA privilege grant'
'DBA privilege grant' is executed by hand by Database Administration, which costs 3.2h and is the source of the recurring configuration errors visible in the ticket data.

*Expected saving:* ~1.2 hour(s) per request.
- Model the target configuration as a versioned template.
- Apply it through the vendor API inside a transaction, with an automatic rollback when validation fails.
- Re-read the applied configuration and attach the diff to the ticket.

```python
"""Provision the target system from a versioned template instead of by hand."""

import requests

FIREWALL_API = "https://firewall.corp.local/api/v3"


def provision(ticket: dict) -> dict:
    """Apply the access template, verify it, and attach the diff to the ticket."""
    payload = {
        "user": ticket["requester_uid"],
        "profile": ticket["access_profile"],       # template, not free-form rules
        "valid_until": ticket["valid_until"],      # every grant is time-boxed
        "ticket": ticket["id"],
    }
    applied = requests.post(f"{FIREWALL_API}/vpn-users", json=payload, timeout=15)
    applied.raise_for_status()

    # Read back and compare: configuration drift is detected immediately.
    current = requests.get(
        f"{FIREWALL_API}/vpn-users/{ticket['requester_uid']}", timeout=15
    ).json()
    if current["profile"] != payload["profile"]:
        rollback(ticket)
        raise RuntimeError(f"provisioning drift for ticket {ticket['id']}")
    return current
```

## 4. Expected effect
- Cycle-time saving: ~21.4 h per request.
- Projected end-to-end duration: **26.0 h -> 4.6 h** (inside the 8 h SLA).

### ۱۲.۲ The systems manager rejects it and redirects the system

The feedback is written into the checkpointed state with `update_state`, and
the graph is resumed with `invoke(None, config)` — "continue from where you
stopped". The router sends it back to the engineer, which re-plans under the
new constraint and stops again for a second approval.

In [41]:
rule("SCENARIO 2 — human feedback and re-planning")

FEEDBACK = (
    "The security approval cannot be removed. "
    "Propose a way to accelerate the security review instead."
)

# Exactly what `WorkflowSession` does internally, shown here step by step.
system.compiled_graph.update_state(
    config, {"approved": False, "human_feedback": FEEDBACK}
)
resumed = system.compiled_graph.invoke(None, config)

print(f"\nnext node after resuming = {system.compiled_graph.get_state(config).next}")
print(f"revision                 = {resumed['revision']}")
print(f"constraints now in force = {resumed['constraints']}")


SCENARIO 2 — human feedback and re-planning
revision 1 REJECTED - feedback: 'The security approval cannot be removed. Propose a way to accelerate the security review instead.'


re-planning after human feedback: 'The security approval cannot be removed. Propose a way to accelerate the security review instead.'


  derived constraint -> preserve control -> Security officer review (accelerate the control instead of removing it)


revision 2: 8 gap(s) identified (high, high, medium, medium, medium, medium, medium, medium)


  [high] End-to-end duration breaches the documented SLA


...

  proposal 'Decision-support fast track for 'Security officer review'' (~14.0h, cycle_time)


  proposal 'Push-based approval bot for 'Direct manager approval'' (~2.5h, cycle_time)


  proposal 'Direct API provisioning for 'DBA privilege grant'' (~1.2h, cycle_time)


projected duration 26h -> 8.3h (SLA 8h)



next node after resuming = ('human_review',)
revision                 = 2
constraints now in force = [{'kind': 'preserve_control', 'target': 'Security officer review', 'directive': 'accelerate the control instead of removing it', 'source_feedback': 'The security approval cannot be removed. Propose a way to accelerate the security review instead.'}]


In [42]:
# The unsafe proposal is gone; the alternative keeps the control and
# accelerates it - the security officer's fast-track dashboard.
for proposal in resumed["action_plan"]["proposals"]:
    flag = "⚠ REMOVES A CONTROL" if proposal["removed_controls"] else "✓ keeps controls"
    print(f"{flag:<22} {proposal['title']}  (~{proposal['expected_gain_hours']}h)")

✓ keeps controls       Decision-support fast track for 'Security officer review'  (~14.0h)
✓ keeps controls       Push-based approval bot for 'Direct manager approval'  (~2.5h)
✓ keeps controls       Direct API provisioning for 'DBA privilege grant'  (~1.2h)


In [43]:
display(Markdown(resumed["proposed_solution"]))

# Action Plan - Privileged database access provisioning (revision 2)

The documented SLA for 'Privileged database access provisioning' is 8h, but the 2024-Q4 (85 tickets) sample shows an average of 26h - 3.2x the standard. The dominant bottleneck is 'Security officer review' with 18h (69% of the cycle, 80% of it pure waiting). 8 gap(s) were identified and 3 automation measure(s) are proposed below. This is revision 2, rewritten after the systems manager's feedback. Constraints taken into account: preserve control -> Security officer review (accelerate the control instead of removing it).

## 1. Identified gaps
- **End-to-end duration breaches the documented SLA** (high)
  - evidence: SOP target is 8h but the measured average is 26h (3.2x the SLA, +18h).
- **Bottleneck at stage 'Security officer review'** (high) [stage: Security officer review]
  - evidence: 'Security officer review' consumes 18h (69% of the total cycle time), of which 80% is pure waiting. SOP target for this step is 4h.
- **Bottleneck at stage 'Direct manager approval'** (medium) [stage: Direct manager approval]
  - evidence: 'Direct manager approval' consumes 3.5h (13% of the total cycle time), of which 80% is pure waiting. SOP target for this step is 1h.
- **Bottleneck at stage 'DBA privilege grant'** (medium) [stage: DBA privilege grant]
  - evidence: 'DBA privilege grant' consumes 3.2h (12% of the total cycle time), of which 45% is pure waiting. SOP target for this step is 2h.
- **Rework rate above the acceptable threshold** (medium)
  - evidence: 9 of 85 tickets were reopened (11%); the accepted ceiling is 10%.
- **Recurring operational error: Business justification too vague, returned by the security officer** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (85 tickets).
- **Recurring operational error: Requested privilege level higher than the peer group baseline** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (85 tickets).
- **Recurring operational error: Expiry date left empty, grant defaulted to permanent** (medium)
  - evidence: Reported repeatedly during 2024-Q4 (85 tickets).

## 2. Proposed automation
### Decision-support fast track for 'Security officer review'
The control stays exactly where the policy puts it (ISMS-POL-07); what is removed is the 18h the officer currently spends *collecting* evidence by hand. Every input the reviewer needs is pre-assembled and pushed to a single queue.

*Expected saving:* ~14 hour(s) per request.
- Compute a risk score at intake (requester role, data classification, requested privilege level, historical incidents).
- Pre-assemble the evidence pack: current entitlements, peer-group comparison, last access review, open incidents.
- Render one real-time dashboard queue for the security officer, ordered by risk and remaining SLA, with one-click approve/reject and a mandatory justification field.
- Auto-expire the granted privilege so that low-risk requests are time-boxed rather than permanent.
- The Security officer review decision itself remains 100% human.
- Controls kept intact: Security officer review

```python
"""Fast-track the security review WITHOUT removing it.

The officer still decides; the script only removes the manual evidence
gathering and orders the queue by risk and remaining SLA.
"""

from dataclasses import dataclass

WEIGHTS = {"privilege": 0.4, "data_class": 0.3, "role_fit": 0.2, "history": 0.1}


@dataclass(frozen=True)
class ReviewItem:
    ticket_id: str
    risk_score: float
    sla_left_hours: float
    evidence: dict


def score(request: dict) -> float:
    """Cheap, explainable risk score - never an automatic approval."""
    signals = {
        "privilege": request["privilege_level"] / 5,
        "data_class": request["data_classification"] / 4,
        "role_fit": 1 - request["peer_group_match"],
        "history": min(request["past_incidents"], 3) / 3,
    }
    return round(sum(WEIGHTS[key] * value for key, value in signals.items()), 3)


def build_queue(requests_: list[dict]) -> list[ReviewItem]:
    """Everything the officer needs, pre-assembled, highest risk first."""
    items = [
        ReviewItem(
            ticket_id=request["id"],
            risk_score=score(request),
            sla_left_hours=request["sla_left_hours"],
            evidence={
                "current_entitlements": fetch_entitlements(request["requester_uid"]),
                "peer_comparison": fetch_peer_group(request["requester_uid"]),
                "last_access_review": fetch_last_review(request["requester_uid"]),
                "open_incidents": fetch_incidents(request["requester_uid"]),
            },
        )
        for request in requests_
    ]
    items.sort(key=lambda item: (-item.risk_score, item.sla_left_hours))
    return items


def record_decision(item: ReviewItem, officer_id: str, approved: bool, note: str) -> None:
    """The human decision is mandatory and always audited."""
    assert note, "a justification is required by the information-security policy"
    audit_log.write(
        ticket=item.ticket_id, officer=officer_id, approved=approved,
        risk_score=item.risk_score, note=note,
    )
```
### Push-based approval bot for 'Direct manager approval'
'Direct manager approval' currently takes 3.5h, almost all of it queue time while the request waits in the approver's inbox. The approval *decision* stays with Direct Manager; only the way the request reaches them and the way the result is applied become automatic.

*Expected saving:* ~2.5 hour(s) per request.
- On ticket creation, post an interactive approve/reject card to the corporate messenger account of Direct Manager.
- Start an SLA timer; auto-escalate to the line manager's deputy when no answer arrives within 1h.
- On approval, call the downstream provisioning API directly instead of creating a second manual task.
- Write the approver identity, timestamp and payload to the audit log so the control remains fully traceable.
- Controls kept intact: Direct manager approval

```python
"""Push-based approval: ask the manager in the messenger, provision on YES."""

import datetime as dt
import requests

MESSENGER_API = "https://messenger.corp.local/api/v1"
SERVICE_DESK_API = "https://servicedesk.corp.local/api/v2"
ESCALATION_AFTER = dt.timedelta(hours=1)


def request_approval(ticket: dict) -> dict:
    """Post an interactive approve/reject card to the approver."""
    response = requests.post(
        f"{MESSENGER_API}/cards",
        json={
            "to": ticket["approver_id"],
            "title": f"Access request {ticket['id']} - {ticket['requester']}",
            "body": ticket["justification"],
            "actions": [
                {"id": "approve", "label": "Approve", "style": "primary"},
                {"id": "reject", "label": "Reject", "style": "danger"},
            ],
            "callback_url": f"{SERVICE_DESK_API}/tickets/{ticket['id']}/decision",
            "expires_at": (dt.datetime.utcnow() + ESCALATION_AFTER).isoformat(),
        },
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


def on_decision(ticket: dict, decision: dict) -> None:
    """Callback invoked by the messenger when the approver clicks a button."""
    audit(ticket, decision)                       # the control stays traceable
    if decision["action"] != "approve":
        close_ticket(ticket, status="rejected", note=decision.get("comment", ""))
        return
    provision(ticket)                             # call the target system API
    close_ticket(ticket, status="done")


def escalate_if_silent(ticket: dict) -> None:
    """Re-route to the deputy when the SLA timer expires (no lost requests)."""
    if ticket["status"] == "awaiting_approval" and _age(ticket) > ESCALATION_AFTER:
        ticket["approver_id"] = ticket["deputy_approver_id"]
        request_approval(ticket)
```
### Direct API provisioning for 'DBA privilege grant'
'DBA privilege grant' is executed by hand by Database Administration, which costs 3.2h and is the source of the recurring configuration errors visible in the ticket data.

*Expected saving:* ~1.2 hour(s) per request.
- Model the target configuration as a versioned template.
- Apply it through the vendor API inside a transaction, with an automatic rollback when validation fails.
- Re-read the applied configuration and attach the diff to the ticket.

```python
"""Provision the target system from a versioned template instead of by hand."""

import requests

FIREWALL_API = "https://firewall.corp.local/api/v3"


def provision(ticket: dict) -> dict:
    """Apply the access template, verify it, and attach the diff to the ticket."""
    payload = {
        "user": ticket["requester_uid"],
        "profile": ticket["access_profile"],       # template, not free-form rules
        "valid_until": ticket["valid_until"],      # every grant is time-boxed
        "ticket": ticket["id"],
    }
    applied = requests.post(f"{FIREWALL_API}/vpn-users", json=payload, timeout=15)
    applied.raise_for_status()

    # Read back and compare: configuration drift is detected immediately.
    current = requests.get(
        f"{FIREWALL_API}/vpn-users/{ticket['requester_uid']}", timeout=15
    ).json()
    if current["profile"] != payload["profile"]:
        rollback(ticket)
        raise RuntimeError(f"provisioning drift for ticket {ticket['id']}")
    return current
```

## 3. Constraints honoured in this revision
- preserve control -> Security officer review (accelerate the control instead of removing it)

## 4. Expected effect
- Cycle-time saving: ~17.7 h per request.
- Projected end-to-end duration: **26.0 h -> 8.3 h** (still above the 8 h SLA).

In [44]:
# Approve revision 2 and let the graph reach END.
system.compiled_graph.update_state(config, {"approved": True, "human_feedback": ""})
final = system.compiled_graph.invoke(None, config)

print(f"\napproved      : {final['approved']}")
print(f"revisions     : {final['revision']}")
print(f"next node     : {system.compiled_graph.get_state(config).next}  (empty = END)")
print(f"feedback log  : {final['feedback_log']}")

revision 2 APPROVED by the systems manager



approved      : True
revisions     : 2
next node     : ()  (empty = END)
feedback log  : ['revision 1: The security approval cannot be removed. Propose a way to accelerate the security review instead.']


### ۱۲.۳ The same scenario, driven end-to-end by the session runner

Everything above is what `WorkflowSession.run()` automates. With a
`ScriptedApprovalGateway` the whole two-revision conversation runs in a single
call — and reproducibly, which is what makes it a regression test.

In [45]:
from process_automation.infrastructure.approval import approve, reject_with

rule("SCENARIO 2 — same conversation, one call")

scripted_system = build_system(
    Settings(),
    approval_gateway=ScriptedApprovalGateway([reject_with(FEEDBACK), approve()]),
)
scenario_2 = scripted_system.session.run(
    "privileged access to a production database", thread_id="scenario-2-scripted"
)

print(f"\napproved   : {scenario_2.approved}")
print(f"revisions  : {scenario_2.revisions}")
print(f"interrupts : {scenario_2.interrupts}")
for index, decision in enumerate(scenario_2.decisions, start=1):
    verdict = "APPROVED" if decision.approved else "REJECTED"
    print(f"  revision {index}: {verdict} {decision.feedback}")


SCENARIO 2 — same conversation, one call
starting analysis of 'privileged access to a production database' on thread 'scenario-2-scripted'


RAG lookup for: 'privileged access to a production database' (top_k=4)


  hit score=0.481 doc=SOP-SEC-021 section='header'


  hit score=0.355 doc=SOP-SEC-021 section='1. Purpose and scope'


  hit score=0.210 doc=SOP-SEC-021 section='2. Service level agreement'


...

projected duration 26h -> 8.3h (SLA 8h)


graph interrupted before 'human_review' (revision 2) - waiting for the human


reviewer decision: APPROVED


revision 2 APPROVED by the systems manager



approved   : True
revisions  : 2
interrupts : 2
  revision 1: REJECTED The security approval cannot be removed. Propose a way to accelerate the security review instead.
  revision 2: APPROVED 


In [46]:
scenario_2.print_trace()

[documentation_agent] RAG lookup for: 'privileged access to a production database' (top_k=4)
[documentation_agent]   hit score=0.481 doc=SOP-SEC-021 section='header'
[documentation_agent]   hit score=0.355 doc=SOP-SEC-021 section='1. Purpose and scope'
[documentation_agent]   hit score=0.210 doc=SOP-SEC-021 section='2. Service level agreement'
[documentation_agent]   hit score=0.179 doc=SOP-SEC-021 section='4. Policy notes'
[documentation_agent] extracted SLA=8h and 5 approved step(s) from SOP-SEC-021
[documentation_agent] mandatory control(s) found: Security officer review / ISMS-POL-07
[diagnostic_agent] calling tool(s) fetch_process_metrics, fetch_stage_timings for process 'Privileged database access provisioning'
[diagnostic_agent] measured average=26h over 85 ticket(s) [2024-Q4 (85 tickets)], reopened=9 (11%)
[diagnostic_agent]   stage 'Request registration': 0.5h (2% of cycle, 10% waiting)
[diagnostic_agent]   stage 'Direct manager approval': 3.5h (13% of cycle, 80% waiting)
[dia

### ۱۲.۴ اجرای تعاملی — reviewing the plan yourself

`ConsoleApprovalGateway` prompts you in the notebook. Uncomment and run it to
be the human in the loop: type `y` to approve, or type your feedback to send
the engineer back to work.

In [47]:
# interactive_system = build_system(Settings(), approval_gateway=ConsoleApprovalGateway())
# interactive_result = interactive_system.session.run(
#     "privileged access to a production database", thread_id="interactive"
# )
# print("approved:", interactive_result.approved, "after", interactive_result.revisions, "revision(s)")

---
## ۱۳. Test suite and design review

### ۱۳.۱ Automated tests

Both scenarios above are also encoded as regression tests, next to unit tests
for every layer.

In [48]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    env={"PYTHONPATH": str(SRC), "PATH": __import__("os").environ["PATH"]},
)
print(completed.stdout[-2000:] or completed.stderr[-2000:])

...............................................................          [100%]
63 passed in 0.52s



### ۱۳.۲ How the SOLID principles shaped this design

| Principle | Where it shows up | What it bought us |
|---|---|---|
| **S** — Single Responsibility | `GapAnalyzer` *detects*, `RemediationRegistry` *prescribes*, `routing.py` *routes*, `WorkflowSession` *drives*, `ApprovalGateway` *asks*. No class does two of these. | The revision loop was re-tuned without reopening a single agent; the router is testable with a dict literal. |
| **O** — Open/Closed | `GapRule` and `RemediationStrategy` are registered, never `if/elif`-ed. `ProcessAnalysisState` *extends* the brief's `WorkflowState`. SOP documents are data files. | A new deviation, a new remedy or a whole new process is added without editing an existing class. |
| **L** — Liskov Substitution | `HashingTfidfEmbeddings` ↔ `LangChainEmbeddingsAdapter`; `AutoApprove` ↔ `Scripted` ↔ `Console` ↔ `Callback` gateways; `RuleBasedSolutionComposer` ↔ `LlmNarrativeComposer`. | The same compiled graph served a scripted run, an interactive run and the test suite in this notebook. |
| **I** — Interface Segregation | `interfaces.py` holds eight *tiny* ports. A `MetricsProvider` implementer knows nothing about embeddings. | Test doubles are three lines long; no class is forced to implement behaviour it does not need. |
| **D** — Dependency Inversion | Agents depend only on ports. Concrete classes are chosen in exactly one file, `application/container.py`. | Moving to FAISS + a real Jira API + a hosted LLM is a one-file change. The domain layer imports no framework at all. |

### ۱۳.۳ Layering

```
            ┌──────────────────────────────────────────────┐
  depends   │  application/   composition root + runner    │
     on     ├──────────────────────────────────────────────┤
     │      │  graph/         nodes, edges, checkpointer   │
     │      │  agents/        the 4 LangGraph nodes        │
     ▼      ├──────────────────────────────────────────────┤
            │  interfaces.py  the ports (abstractions)     │
            ├──────────────────────────────────────────────┤
            │  domain/        business rules — no framework│
            └──────────────────────────────────────────────┘
               infrastructure/  adapters, implementing the
                                ports from the outside in
```

Dependency arrows point **inwards**: `domain/` imports nothing from the
project, and `infrastructure/` is imported by the composition root alone.

### ۱۳.۴ Design decisions worth defending

* **The numbers are deterministic, the prose is optional.** SLA breaches,
  bottleneck shares and expected savings come from auditable rules. A language
  model is attached as a *decorator* that rewrites the management summary only,
  so an outage — or a hallucination — can never alter a figure in a compliance
  document.
* **The engineer is aggressive on purpose.** `allow_control_removal=True`
  reproduces the brief's second scenario. Flipping that one setting yields a
  policy-compliant engineer that never proposes dropping a documented control
  (there is a test for it).
* **Constraints are sticky.** A limit stated in revision 1 still holds in
  revision 3; that is what makes the feedback loop converge instead of
  oscillating.
* **Honest projections.** Rework savings are marked `amortised` and are
  excluded from the single-request cycle-time projection, so the plan never
  promises 42 h → −1 h.

### ۱۳.۵ Running with a hosted language model

```bash
pip install langchain-openai        # or langchain-anthropic
export OPENAI_API_KEY=...
export PA_REASONING_MODE=llm        # 'auto' (default), 'llm' or 'offline'
```

`build_language_model()` then returns a chat model and the container decorates
the composer with `LlmNarrativeComposer`. Nothing else in the system changes —
which is the whole point.

---

## پیوست — Delivery checklist

| Requirement of section ۵ (نحوه تحویل پروژه) | Where |
|---|---|
| فایل نوت‌بوک شامل کدهای کامل سیستم چندعاملی | this notebook — every mandated class printed from source |
| نمایش گراف با `.get_graph().draw_mermaid_png()` | §۱۰.۲ |
| لاگ‌های استخراج از RAG | §۱۱.۱ and §۱۱.۴ |
| لاگ‌های فراخوانی Tools | §۴, §۶ and §۱۱.۴ |
| توقف گراف برای دریافت تاییدیه انسانی | §۱۲.۱ (`next = ('human_review',)`) |
| سناریوی اول — گلوگاه و اتوماسیون موفق | §۱۱ |
| سناریوی دوم — مداخله انسانی و اصلاح مسیر | §۱۲ |